# Day2 - Silver: Movie Ratings Summary

**Objetivo:** Criar a tabela Silver `movie_ratings_summary` agregando avaliações por filme a partir das tabelas Bronze `movies` e `ratings`.

| Item | Valor |
|---|---|
| **Camada** | Silver |
| **Entrada** | `{catalog}.bronze.movies`, `{catalog}.bronze.ratings` |
| **Saída** | `{catalog}.silver.movie_ratings_summary` |
| **Ordem no pipeline** | Após `Day2-CreateTablesMovieLens` (Bronze) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Agregação Silver

Faz um JOIN entre `bronze.movies` e `bronze.ratings` e agrega total de avaliações e média de rating por filme. `CREATE OR REPLACE TABLE` garante idempotência — reexecutar sempre recria com os dados mais recentes da Bronze.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`silver`.`movie_ratings_summary` AS
    SELECT
        m.movieId,
        m.title,
        m.genres,
        COUNT(r.rating) AS total_avaliacoes,
        AVG(r.rating) AS media_rating
    FROM
        `${catalog}`.`bronze`.`movies` AS m
            JOIN `${catalog}`.`bronze`.`ratings` AS r
                ON (m.movieId = r.movieId)
    GROUP BY m.movieId, m.title, m.genres;